In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import argparse
from pathlib import Path
from argparse import Namespace

from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from train_sm import MakeDataset, create_data_loaders, EarlyStopping
from sm import NeuralNet

args = Namespace(
    data_dir=Path(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/data_uni"),
    num_batches=200,
    batch_size=512,
    x_train_mean=None,
    x_train_std=None,
    y_train_mean=None,
    y_train_std=None,
    normalize=False,
    layer_dims=[7, 256, 256, 128, 64, 32, 16, 1],
    num_epochs=1_000,
    lr=1e-4,
    save_name=Path(f"/home/hd/hd_hd/hd_gy283/kmc_project/models/sm_uni_1e7"),
)
print(args)

cuda
Namespace(data_dir=PosixPath('/gpfs/bwfor/work/ws/hd_gy283-my_data/data_uni'), num_batches=200, batch_size=512, x_train_mean=None, x_train_std=None, y_train_mean=None, y_train_std=None, normalize=False, layer_dims=[7, 256, 256, 128, 64, 32, 16, 1], num_epochs=1000, lr=0.0001, save_name=PosixPath('/home/hd/hd_hd/hd_gy283/kmc_project/models/sm_uni_1e7'))


In [2]:
train_loader, test_loader, x_train_mean, x_train_std, y_train_mean, y_train_std = create_data_loaders(args.data_dir, args.num_batches, args.batch_size, bool(args.normalize))
args.x_train_mean = x_train_mean
args.x_train_std = x_train_std
args.y_train_mean = y_train_mean
args.y_train_std = y_train_std
print(args)

FileNotFoundError: [Errno 2] No such file or directory: '/gpfs/bwfor/work/ws/hd_gy283-my_data/data_uni/batch_73.npz'

In [15]:
model = NeuralNet(layer_dims=args.layer_dims).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=200, gamma=0.1)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2, eta_min=1e-6)

train_losses = []
val_losses = []

early_stopping = EarlyStopping(patience=10, min_delta=0.001)

for epoch in range(1, args.num_epochs+1):

    model.train()
    running_loss = 0.0
    for inputs_batch, targets_batch in train_loader:
        inputs_batch  = inputs_batch.to(device, non_blocking=True)
        targets_batch = targets_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(inputs_batch)
        loss  = criterion(preds, targets_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs_batch.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs_batch, targets_batch in test_loader:
            inputs_batch  = inputs_batch.to(device, non_blocking=True)
            targets_batch = targets_batch.to(device, non_blocking=True)

            preds = model(inputs_batch)
            loss  = criterion(preds, targets_batch)
            val_loss += loss.item() * inputs_batch.size(0)

    epoch_val_loss = val_loss / len(test_loader.dataset)
    #scheduler.step()

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    #current_lr = optimizer.param_groups[0]['lr']
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}/{args.num_epochs} "
              f"   Train Loss: {epoch_train_loss:.10f}"
              f"   Val Loss: {epoch_val_loss:.10f}", flush=True)
    #early_stopping(epoch_val_loss)
    if early_stopping.early_stop:
        print("[EARLY STOPPING TRIGGERED]")
        break

torch.save({
    "model_state_dict": model.state_dict(),
    "args": vars(args),
    "train_losses": train_losses,
    "val_losses": val_losses
}, f"{args.save_name}.pth")

Epoch 10/1000    Train Loss: 0.0108019465   Val Loss: 0.0102590275
Epoch 20/1000    Train Loss: 0.0069599341   Val Loss: 0.0080921998
Epoch 30/1000    Train Loss: 0.0052751559   Val Loss: 0.0055269926
Epoch 40/1000    Train Loss: 0.0043131275   Val Loss: 0.0042992328
Epoch 50/1000    Train Loss: 0.0037168395   Val Loss: 0.0039745148
Epoch 60/1000    Train Loss: 0.0032731250   Val Loss: 0.0035673725
Epoch 70/1000    Train Loss: 0.0029739648   Val Loss: 0.0031866822
Epoch 80/1000    Train Loss: 0.0026558584   Val Loss: 0.0035104208
Epoch 90/1000    Train Loss: 0.0024462331   Val Loss: 0.0026792182
Epoch 100/1000    Train Loss: 0.0022979198   Val Loss: 0.0025403741
Epoch 110/1000    Train Loss: 0.0021414512   Val Loss: 0.0023972134
Epoch 120/1000    Train Loss: 0.0020061894   Val Loss: 0.0022571526
Epoch 130/1000    Train Loss: 0.0018904978   Val Loss: 0.0022631600
Epoch 140/1000    Train Loss: 0.0017982716   Val Loss: 0.0020832731
Epoch 150/1000    Train Loss: 0.0017117039   Val Loss: 0.